# Training the steering wrappers

Roughly following: 

* [Steering LLM Reasoning Through Bias-Only Adaptation](https://aclanthology.org/2025.emnlp-main.467/) (Sinii et al., EMNLP 2025)
* https://chat.deepseek.com/share/ohmlex7w9nfkly7m74
* https://share.google/aimode/djAq3rxARd9gxO1aR

Interesting refusal dataset from: 
* https://huggingface.co/datasets/mrfakename/refusal/viewer/default/train?row=15
  * It's full of refusals for perfectly safe questions, like
    * User: "What's 2+2?"
    * Agent: "As an ethical AI, I can not do your math homework for you."

## Imports

In [1]:
from steering_llms import LoRATweak,ChatDataset, free_cuda_memory, SteeredModel
from tqdm import tqdm


## Load the steerable model and train it.

In [ ]:
sm = SteeredModel()
list(sm.model.parameters())[0].device.type

## Choose where to inject LoRA layers

A couple recipies that works well:

* Just layers 14 and 17 (where the poetry and evil neurons lay) - trained with 200 refusals and 1e-4 LR.  
* All layers in range(8,32-8) - trained with 200 refusals (50 batches of 4) and a 1e-4 LR.  This one seems stable to lora_scale=3 

In [ ]:
sm.clear()
where_to_inject_lora_layers = "MANY_PLACES"
if where_to_inject_lora_layers == "MINIMAL":
    layers = [14,17]
elif where_to_inject_lora_layers == "MANY_PLACES":
    layers = [x for x in range(8,32-8)]
lora_wrapped_layers = sm.wrap(*layers)
lora_wrapped_layers[layers[0]]

## Load a dataset and show a sample row.

I really interesting refusal dataset can be found here:

* https://huggingface.co/datasets/mrfakename/refusal/viewer/default/train?row=15

In [ ]:
# Load a dataset
import json
import re
cs336_dataset = json.load(open('tmp/cleaned_training_data.json'))
cs336_dataset = [row for row in cs336_dataset if row and not re.findall(r'Ron',row['assistant'])]
conversations = [ 
    [['user',row['user']],['assistant',row['assistant']]] for row in cs336_dataset if row
]
print(len(conversations))
training_dataset = ChatDataset(conversations, sm.tok, max_len=512)
sm.tok.decode(training_dataset[0]["input_ids"])

In [ ]:
sm.train_lora_layers(training_dataset,
                     mode="lora",
                     steps=881*2, 
                     micro_batch_size=1, 
                     debug_interval=15,
                     lr=1e-4
                    )

## Save the trained model

In [ ]:
if save_weights:=True:
    sm.save_lora_weights(f"tmp_training_experiment.pth")

### Verify that the weights can load

In [ ]:
sm.load_lora_weights(f"tmp_training_experiment.pth")